# Mehrotra, Levenstein et al. (2024) on dandiset 000987: retrosplenial cortex

Paper: Mehrotra, Levenstein, Duszkiewicz, Skromne Carrasco, Booker, Kwiatkowska and Peyrache (2024),
"Hyperpolarization-activated currents drive neuronal activation sequences in sleep", Current Biology
34, 3043-3054, [doi:10.1016/j.cub.2024.05.048](https://doi.org/10.1016/j.cub.2024.05.048).

Dataset: [DANDI:000987](https://dandiarchive.org/dandiset/000987/0.261003.2342), version
`0.261003.2342`, streamed from the archive. The paper cites the dandiset without a version. Its
published versions (0.260831.1616, 0.260831.1620 and 0.261003.2342) hold the same 14 assets (same
asset ids and sizes); they differ only in the dandiset metadata (release note "Improved metadata").
The most recent published version is pinned, as the protocol asks when the paper cites none.

Claims file: `claims/000987/10.1016-j.cub.2024.05.048.yaml`, written from the version of record
with its STAR Methods and Supplemental Information. Method details come from the claims file, which
cites the paper's code (github.com/PeyracheLab/MehrotraLevenstein_2023 at commit 4ee34f3); where a
detail was taken from the paper by this attempt, the cell says which passage.

This notebook tests the two claims on this dandiset. The claims on the PoSub data (DANDI:000939)
are in `posub_000939.ipynb`. The others are blocked (see the attempt record).

| Claim | Data | Here |
|---|---|---|
| rsc-dataset | this dandiset | tested |
| fig5f-rsc-pmr-onset | this dandiset | tested |
| fig5c-posub-pmr-onset, fig5d-isomap-pmr, fig1e-delay-range, fig1g-dv-sequence-du, fig1h-ventral-down-longer, figs1i-du-more-stable, figs1kl-not-firing-rate | DANDI:000939 | tested in `posub_000939.ipynb` |
| fig1b-no-dv-tuning-gradient, fig1j-down-resets-hd, fig1k-hd-stable-at-onset, figs1j-dorsal-ventral-same-hd, fig2f-gamma-travels, figs2e-speed-similar, figs2d-lfp-spike-agree, figs2g-adn-synchronous, fig5e-fc-pmr-onset | elsewhere | blocked |
| fig3-other-gradients-fail, fig3f-ih-gradient | model code (MATLAB) | blocked |
| fig4d-sag-gradient, figs4-no-other-gradients | ex vivo, not deposited | blocked |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, mannwhitneyu
import pynapple as nap
import mehrotra as m

print("pynapple", nap.__version__, "| dandiset", m.RSC_DANDISET, m.RSC_VERSION)
assets = m.list_assets()
sessions = m.load_all_rsc()
print(len(assets), "assets;", len(sessions), "sessions loaded")

## Data check

Each file holds the spike times of sorted units (no waveforms, no cell type), the behavioural
epochs (`sleep`, `wake`), the NREM epochs of the sleep scoring (`SWS`), the stored `DOWN` and `UP`
intervals, and position and head direction. The dandiset description also lists LFP; the files hold
none (checked below). The electrode tables have x, y and z all zero.

In [ ]:
rows = []
for s in sessions:
    sp = s["spikes"]
    wake = s["epochs"]["wake"]; sleep = s["epochs"]["sleep"]
    rows.append(dict(
        session=s["identifier"], subject=s["subject"], genotype=s["genotype"], units=len(sp),
        spikes=int(sum(len(sp[u]) for u in sp.keys())),
        unit_locations=",".join(sorted(set(s["unit_location"]))),
        sleep_s=round(sleep.tot_length()), wake_s=round(wake.tot_length()),
        nrem_s=round(s["sws"].tot_length()),
        nrem_in_sleep=float(s["sws"].intersect(sleep).tot_length() / s["sws"].tot_length()),
        n_down=len(s["down"]), n_up=len(s["up"]),
        down_ms_median=1000 * float(np.median(s["down"]["end"] - s["down"]["start"])),
        down_ms_range=(round(1000 * float(np.min(s["down"]["end"] - s["down"]["start"]))),
                       round(1000 * float(np.max(s["down"]["end"] - s["down"]["start"])))),
        down_in_nrem=float(s["down"].intersect(s["sws"]).tot_length() / s["down"].tot_length()),
        pos_first=s["position_range"][0], pos_last=s["position_range"][1],
        pos_in_wake=bool(np.all(np.isfinite(np.asarray(wake.in_interval(nap.Ts(t=np.array(s["position_range"][:2]))), dtype=float)))),
        unsorted_units=s["spike_unsorted"],
    ))
check = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(check.to_string())
print("\nsessions", len(check), "| subjects", check.subject.nunique(), dict(check.groupby("subject").size()),
      "| units", check.units.sum())
print("genotypes by subject:", dict(check.groupby("subject").genotype.first()))

In [ ]:
# Files hold no LFP although the dandiset description lists it.
for a in assets[:1] + assets[-1:]:
    h = m.open_asset(a)
    groups = []
    h.visit(lambda name: groups.append(name) if name.split("/")[-1] in ("data",) else None)
    print(a["path"], "| datasets named data:", groups,
          "| processing modules:", list(h["processing"].keys()))

# Recordings deposited more than once: compare every pair of sessions by the spike counts of their
# units and by their first spike times.
sig = {s["identifier"]: sorted(len(s["spikes"][u]) for u in s["spikes"].keys()) for s in sessions}
first = {s["identifier"]: np.sort(np.concatenate([s["spikes"][u].t[:5] for u in s["spikes"].keys()]))[:50]
         for s in sessions}
ids = list(sig)
dups = [(a, b) for i, a in enumerate(ids) for b in ids[i + 1:]
        if sig[a] == sig[b] or np.array_equal(first[a], first[b])]
shared_units = sum(len(set(sig[a]) & set(sig[b])) for i, a in enumerate(ids) for b in ids[i + 1:])
print("pairs with identical unit spike counts or identical first spikes:", dups)
print("unit spike counts shared between sessions (any pair):", shared_units, "of", sum(len(v) for v in sig.values()))
print("session_start_time of the files (all on 2024-05-09) against the identifier dates:")
for a, s in zip(assets, sessions):
    print("  ", a["path"].split("_ses-")[1][:15], s["identifier"])

In [ ]:
# NREM (SWS) epochs against the behavioural epochs.
for s in sessions:
    out = s["sws"].set_diff(s["epochs"]["sleep"])
    if out.tot_length() > 0:
        longest = out[np.argmax(out["end"] - out["start"])]
        print(f'{s["identifier"]}: {out.tot_length():.0f} s of SWS outside the sleep epochs, in {len(out)} intervals; '
              f'longest {float(longest["start"][0]):.1f} to {float(longest["end"][0]):.1f} s; '
              f'wake epochs {[(round(a), round(b)) for a, b in zip(s["epochs"]["wake"]["start"], s["epochs"]["wake"]["end"])]}')

The DOWN and UP intervals stored in the files are the study's, according to the dandiset
description. Two checks follow. (1) They share the clock of the spikes: the population rate inside
the stored DOWN states is compared with the rate in the same intervals shifted by 0.5 s and 1 s, and
with the NREM mean. (2) They are what detectDOWNs_RSC.py gives: DOWN and UP states are re-detected
from the spikes with the code's procedure (`mehrotra.detect_down_states`) and matched interval by
interval (start and end within 10 ms, one bin).

In [ ]:
clock = []
redetected = {}
for s in sessions:
    sp, sws, down = s["spikes"], s["sws"], s["down"]
    nrem_rate = sum(len(sp[u].restrict(sws)) for u in sp.keys()) / sws.tot_length()
    row = dict(session=s["identifier"], nrem_pop_rate=nrem_rate)
    for shift in (0.0, 0.5, 1.0):
        ep = nap.IntervalSet(start=down["start"] + shift, end=down["end"] + shift).intersect(sws)
        row[f"rate_in_down_shift_{shift}s"] = sum(len(sp[u].restrict(ep)) for u in sp.keys()) / ep.tot_length()
    d, u = m.detect_down_states(sp, sws)
    redetected[s["identifier"]] = (d, u)
    md = m.interval_match(down, d)
    mu = m.interval_match(s["up"], u)
    row.update(n_down_stored=len(down), n_down_redetected=len(d), down_matched=md["fraction"],
               up_matched=mu["fraction"],
               redetected_matched_by_stored=m.interval_match(d, down)["fraction"])
    clock.append(row)
clock = pd.DataFrame(clock)
print(clock.round(3).to_string())
print("\nratio of population rate in stored DOWN states to NREM mean: median",
      round(float((clock["rate_in_down_shift_0.0s"] / clock.nrem_pop_rate).median()), 3),
      "range", round(float((clock["rate_in_down_shift_0.0s"] / clock.nrem_pop_rate).min()), 3), "to",
      round(float((clock["rate_in_down_shift_0.0s"] / clock.nrem_pop_rate).max()), 3))
print("same intervals shifted by 0.5 s: median ratio",
      round(float((clock["rate_in_down_shift_0.5s"] / clock.nrem_pop_rate).median()), 3))

## Claim rsc-dataset

> The RSC dataset holds 570 units recorded in 14 sessions from 3 mice.

Criterion: counts of the dandiset's sessions, distinct subjects and sorted units over all sessions;
reproduced when all three match (tolerance zero), partially reproduced when sessions and mice match
and the unit count is within 10% (513 to 627).

The file count alone does not show which units the paper counted, so the unit count is also given
under the selections the paper describes for units (viable units with at least 0.5 Hz in open field,
STAR Methods, Spike sorting and unit classification; units below 10 Hz in open field, the rate half
of the EX rule; units entering the Figure 5F test). The wake epochs are taken as the open field
exploration.

In [ ]:
n_sessions = len(sessions)
n_mice = len({s["subject"] for s in sessions})
n_units = int(sum(len(s["spikes"]) for s in sessions))
print(f"sessions {n_sessions} (expected 14) | mice {n_mice} (expected 3) | units {n_units} (expected 570)")
print(f"unit count relative to 570: {n_units / 570:.3f} ({n_units - 570:+d}); 10% band 513 to 627")

wake_rates = pd.concat({s["identifier"]: m.rate_in(s["spikes"], s["epochs"]["wake"]) for s in sessions})
nrem_rates = pd.concat({s["identifier"]: m.rate_in(s["spikes"], s["sws"]) for s in sessions})
sel = pd.DataFrame(dict(
    all=wake_rates.groupby(level=0).size(),
    wake_at_least_0p5Hz=(wake_rates >= 0.5).groupby(level=0).sum(),
    wake_below_10Hz=(wake_rates < 10).groupby(level=0).sum(),
    wake_0p5_to_10Hz=((wake_rates >= 0.5) & (wake_rates < 10)).groupby(level=0).sum(),
    nrem_rate_above_0=(nrem_rates > 0).groupby(level=0).sum(),
))
print(sel.to_string())
print("totals:", dict(sel.sum()))
print("units per session, range:", sel["all"].min(), "to", sel["all"].max(), "(description: 31 to 71)")

**Result: not reproduced.** The dandiset holds 14 sessions from 3 mice, as the paper states, but 638
sorted units, not 570: 68 more (1.119 times), outside the 10% band (513 to 627) of the criterion. None
of the unit selections of the Methods gives 570 (606 units with at least 0.5 Hz in wake, 412 below
10 Hz, 380 both). The units per session range from 31 to 71, as the dandiset description says. The
next section shows that 570 is the number of units that enter the Figure 5F test.

## Claim fig5f-rsc-pmr-onset

> In retrosplenial cortex, as in PoSub, units with a larger firing-rate overshoot at UP state onset
> (peak-to-mean ratio, PMR) enter the UP state earlier; UP onset delay and PMR are negatively
> correlated (Kendall tau = -0.28, n = 570 units from 14 sessions in 3 mice, p = 6.76e-21).

Criterion (claims file): the method of Figure5F.py and detectDOWNs_RSC.py, implemented in
`mehrotra.du_peth` and `mehrotra.pmr_onset`; the stored UP states as the study's, with the
re-detected ones as a check; units pooled over sessions and Kendall's tau between UP onset delay
and PMR. Two readings of which units enter: (a) all sorted units, (b) putative EX units by
trough-to-peak above 0.35 ms and open field rate below 10 Hz. Reproduced when tau is negative with
p < 0.05 under both readings and within 0.028 of -0.28 under at least one.

Reading (b) cannot be applied as written: the files hold no waveforms, so trough-to-peak is unknown.
The rate half of the rule (open field rate below 10 Hz, with and without the 0.5 Hz viability
floor) is run in its place and labelled as such.

In [ ]:
results = []
peths = {}
for s in sessions:
    sid = s["identifier"]
    for source, up in (("stored", s["up"]), ("redetected", redetected[sid][1])):
        p = m.du_peth(s["spikes"], up)
        r = m.pmr_onset(p)
        r["session"] = sid; r["subject"] = s["subject"]; r["up_source"] = source
        r["wake_rate"] = wake_rates[sid].reindex(r.index).values
        results.append(r.reset_index())
        if source == "stored":
            peths[sid] = p
res = pd.concat(results, ignore_index=True)

def tau_row(df, label):
    d = df[df.entered]
    t, p = kendalltau(d.onset, d.pmr)
    return dict(reading=label, units_considered=len(df), units_entered=len(d),
                units_dropped=int((~df.entered).sum()), tau=round(t, 4), p=p,
                within_tolerance=abs(t - (-0.28)) <= 0.028)

st = res[res.up_source == "stored"]
rd = res[res.up_source == "redetected"]
# Variant: only UP states that start inside the sleep epochs (some SWS lies in wake epochs).
rows_sleep = []
for s in sessions:
    up = s["up"]
    keep = np.asarray(np.isfinite(np.asarray(s["epochs"]["sleep"].in_interval(nap.Ts(t=up["start"])), dtype=float)))
    up_s = nap.IntervalSet(start=up["start"][keep], end=up["end"][keep])
    r = m.pmr_onset(m.du_peth(s["spikes"], up_s))
    rows_sleep.append(r)
in_sleep = pd.concat(rows_sleep, ignore_index=True)
table = pd.DataFrame([
    tau_row(st, "(a) all units, stored UP states"),
    tau_row(rd, "(a) all units, re-detected UP states"),
    tau_row(st[st.wake_rate < 10], "(b, rate half only) wake rate < 10 Hz, stored"),
    tau_row(st[(st.wake_rate < 10) & (st.wake_rate >= 0.5)], "(b, rate half only) 0.5 <= wake rate < 10 Hz, stored"),
    tau_row(rd[rd.wake_rate < 10], "(b, rate half only) wake rate < 10 Hz, re-detected"),
    tau_row(in_sleep, "(a) all units, stored UP states starting inside the sleep epochs"),
])
print(table.to_string())
print("\nunits with wake rate >= 10 Hz (left out by the rate half of reading b):",
      int((st.wake_rate >= 10).sum()), "of", len(st))
print("UP onset delay range (s):", st.onset.min(), "to", st.onset.max(),
      "| PMR range:", round(st.pmr.min(), 3), "to", round(st.pmr.max(), 3))

In [ ]:
# By mouse and by session (stored UP states, all units).
by = []
for key, col in (("subject", "subject"), ("session", "session")):
    for k, d in st.groupby(col):
        e = d[d.entered]
        t, p = kendalltau(e.onset, e.pmr) if len(e) > 2 else (np.nan, np.nan)
        by.append(dict(level=key, name=k, units=len(d), entered=len(e), tau=round(t, 3), p=p))
by = pd.DataFrame(by)
print(by.to_string())
ses = by[by.level == "session"]
print("\nsessions with negative tau:", int((ses.tau < 0).sum()), "of", len(ses),
      "| with negative tau and p < 0.05:", int(((ses.tau < 0) & (ses.p < 0.05)).sum()))

# Sensitivity of the pooled result to the onset threshold (the paper fixes 0.5; shown as a check).
sens = []
for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
    rr = pd.concat([m.pmr_onset(p, threshold=thr) for p in peths.values()])
    e = rr[rr.entered]
    t, pv = kendalltau(e.onset, e.pmr)
    sens.append(dict(threshold=thr, entered=len(e), of=len(rr), tau=round(t, 4), p=pv))
print(pd.DataFrame(sens).to_string())

In [ ]:
e = st[st.entered]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
H, xe, ye = np.histogram2d(e.onset, e.pmr, bins=[np.arange(-0.0025, 0.16, 0.005), np.linspace(0.5, 3.6, 32)])
ax[0].imshow(np.ma.masked_equal(H.T, 0), origin="lower", aspect="auto", extent=[xe[0], xe[-1], ye[0], ye[-1]])
mm, bb = np.polyfit(e.onset, e.pmr, 1)
xx = np.array([0, 0.155]); ax[0].plot(xx, mm * xx + bb, "r")
ax[0].set_xlabel("UP onset delay (s)"); ax[0].set_ylabel("PMR"); ax[0].set_title("RSC, all units, stored UP states")
print("units outside the PMR axis (0.5 to 3.6):", int(((e.pmr < 0.5) | (e.pmr > 3.6)).sum()), "of", len(e))
allp = pd.concat(peths.values(), axis=1)
order = np.argsort(st.set_index(["session", "unit"]).onset.values)
ax[1].imshow(allp.loc[-0.25:0.25].T.values[order], aspect="auto", vmin=0, vmax=2,
             extent=[-0.25, 0.25, len(order), 0])
ax[1].set_xlabel("time from DU (s)"); ax[1].set_ylabel("unit (sorted by onset)")
ax[1].set_title("normalized DU-PETH (colour 0 to 2)")
plt.tight_layout(); plt.show()
print("normalized PETH range over -0.25 to 0.25 s:", round(float(np.nanmin(allp.loc[-0.25:0.25].values)), 3),
      "to", round(float(np.nanmax(allp.loc[-0.25:0.25].values)), 3))

**Result: reproduced, resting on an assumption.** With all sorted units (reading a) and the stored UP
states, 570 of 638 units cross half their mean rate within 155 ms of DU, the n of the Figure 5F legend,
and Kendall tau between UP onset delay and PMR is -0.275 (p = 8.1e-21; paper -0.28, p = 6.76e-21),
within the tolerance of 0.028. The re-detected UP states give -0.274 (568 units) and the UP states that
start inside the sleep epochs -0.281 (574 units). Reading (b) cannot be applied as written, as the files
hold no waveforms; its rate half gives tau from -0.342 to -0.329 (p below 1e-18), negative and
significant but outside the tolerance. Tau is negative with p < 0.05 in each of the 14 sessions and in
each mouse (-0.42 to -0.33). It stays between -0.26 and -0.24 for onset thresholds of 0.3, 0.4, 0.6 and
0.7. The outcome rests on reading (a), which the exact count of 570 entering units supports.

## Outside data for two blocked claims

Two claims use data the paper names outside this dandiset, and are blocked (see the attempt
record). The checks behind the blocks are run here.

- fig5e-fc-pmr-onset uses the rat frontal cortex data of Watson et al., CRCNS fcx-1. The data set
  is 27 gzip-compressed tar archives; the members the analysis needs (spike times, cell classes, UP
  states) sit in each archive beside the wide-band `.eeg` file, and a gzip stream can only be read
  from its start. The cell below measures the rate of streaming and decompressing the first 300 MB
  of one archive and gives the time to read all archives at that rate, the bound when the needed
  members come after the `.eeg`. The MD5 sums of the archives, from the data set's
  `checksums.md5`, are printed so that a change is detectable.
- figs2g-adn-synchronous uses the ADN recordings of Peyrache et al. 2015, for which the paper names
  no archive. The sessions the paper's code names (its event files `Mouse12-120806` and so on) were
  looked for in DANDI:000056 (Peyrache et al. 2015, version 0.250624.0430); the cell lists which are
  there and what its units table holds.

In [ ]:
sizes = m.crcns_sizes()
md5 = m.fcx_checksums()
tf = m.time_fcx_fetch()
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in tf.items()})
total = sum(sizes.values())
print(f"fcx-1: {len(sizes)} archives, {total / 1e9:.1f} GB compressed; at {tf['rate_MB_s']:.1f} MB/s, reading all takes "
      f"{total / 1e6 / tf['rate_MB_s'] / 60:.0f} min (notebook limit 60 min)")
for k in sorted(sizes):
    print(f"  {k}  {sizes[k] / 1e9:.2f} GB  md5 {md5.get(k, '?')}")

adn = ["Mouse12-120806", "Mouse12-120807", "Mouse12-120808", "Mouse12-120809", "Mouse12-120810",
       "Mouse17-130129", "Mouse17-130130", "Mouse20-130517", "Mouse32-140822"]
a56 = m.list_assets("000056", "0.250624.0430")
have = {a["path"].split("_ses-")[1].split("_")[0]: a for a in a56}
print("\nADN sessions of the paper's code found in DANDI:000056:", [x for x in adn if x in have],
      "| missing:", [x for x in adn if x not in have])
h = m.open_asset(have["Mouse12-120807"])
print("units table columns:", list(h["units"].keys()))
print("electrode locations:", sorted(set(x.decode() for x in h["general/extracellular_ephys/electrodes/location"][:])))

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

All claims of the claims file, with their outcomes; the claims on DANDI:000939 (`data: elsewhere`)
are in `posub_000939.ipynb` and `posub_lfp_000939.ipynb` and are counted apart from this dandiset.

| Claim | Data | Outcome |
|---|---|---|
| rsc-dataset | 000987 | not reproduced (638 units, not 570) |
| fig5f-rsc-pmr-onset | 000987 | reproduced, resting on an assumption (tau = -0.275, n = 570) |
| fig5c-posub-pmr-onset | 000939 | reproduced, resting on an assumption |
| fig5d-isomap-pmr | 000939 | reproduced |
| fig1e-delay-range | 000939 | reproduced |
| fig1g-dv-sequence-du | 000939 | reproduced, resting on an assumption |
| fig1h-ventral-down-longer | 000939 | reproduced |
| fig1b-no-dv-tuning-gradient | 000939 | reproduced, resting on an assumption |
| figs1j-dorsal-ventral-same-hd | 000939 | reproduced |
| figs1i-du-more-stable | 000939 | blocked, methods underspecified |
| figs1kl-not-firing-rate | 000939 | blocked, methods underspecified |
| fig2f-gamma-travels | 000939 | not reproduced |
| figs2d-lfp-spike-agree | 000939 | blocked, methods underspecified |
| figs2e-speed-similar | 000939 | not reproduced, resting on an assumption |
| fig1j-down-resets-hd, fig1k-hd-stable-at-onset | 000939 | blocked, not attempted in the run's time |
| figs2g-adn-synchronous | 000939 and ADN data | blocked, metadata insufficient |
| fig5e-fc-pmr-onset | CRCNS fcx-1 | blocked, not runnable (fetch over the limit) |
| fig3-other-gradients-fail, fig3f-ih-gradient | model code | blocked, not runnable (MATLAB) |
| fig4d-sag-gradient, figs4-no-other-gradients | ex vivo | blocked, data missing |

How the analysis differs from the paper's. The PETH, PMR and onset delay follow Figure5F.py with
pynapple 0.11 in place of the unpinned version the code used. The paper's unit selection for Figure
5F is not stated; all sorted units are used, as Figure5F.py does, and the count of entering units
(570) matches the legend. The EX reading cannot be run without waveforms.

What was found about the dandiset (numbers above):

- It holds 638 units; the paper's Methods and the dandiset description say 570 were recorded, which is
  the number of units entering the Figure 5F test.
- The description lists LFP; the files hold none. Electrode positions (x, y, z) are all zero, and the
  units carry no waveform, cell type or channel, so the EX rule of the Methods cannot be applied.
- `session_start_time` is the upload time (2024-05-09) for every file, while the identifiers carry the
  recording dates (2021).
- In 9 sessions part of the NREM (`SWS`) epochs lies outside the sleep epochs; in A7803-210730 one SWS
  interval covers two whole wake epochs (1,885 s).
- The stored DOWN and UP states are those detectDOWNs_RSC.py gives (95% to 99% of DOWN states matched
  within 10 ms after re-detection), and the population rate inside them is a median 0.038 of the NREM
  mean (0.95 for the same intervals shifted by 0.5 s), so they share the clock of the spikes.